# تدريب نماذج MedMNIST على Colab — نفس وصفة `train_medmnist_v2.py`

هذا الدفتر **ينقل الوصفة نفسها** الموجودة بـ`api/train_medmnist_v2.py` إلى بطاقة Colab،
ويحفظ الـcheckpoint **بنفس الصيغة بالضبط** الي يقراها المشروع — يعني الملف الناتج ينزل
مباشرة بـ`api/models/` ويشتغل مع `verify_retrain_gains.py` و`tune_threshold.py` بدون أي تعديل.

## ليش هذا الدفتر موجود

جهاز التطوير عنده **٨ غيغا رام** و**٤ غيغا VRAM**، وهذي المهام **فشلت عليه فعلياً** (موثّقة
بـ`TRAINING_LOG.md`):

| المهمة | شنو صار محلياً | على T4 |
|---|---|---|
| `oct` / `oct_bin` (٩٧ ألف صورة) | `MemoryError` — memmap ١.٦ غيغا | ✅ |
| ensembles ٥ بذور | ٣ من ٥ سقطن، والباقي أخذ ٢١.٨ ساعة | ✅ |
| جذوع أقوى (ConvNeXt / EfficientNet) | ما تدخل بـ٤ غيغا | ✅ |
| دقة ٣٨٤–٤٤٨ | مستحيلة | ✅ |

## ⚠️ قاعدتان غير قابلتان للتفاوض

1. **أي رقم يطلع من هنا لازم يُعاد قياسه محلياً** عبر `verify_retrain_gains.py` قبل ما ينُنشر.
   السبب مسجّل بالسجل: رقم من مسار تقييم ثاني مو قابل للمقارنة. **كولاب تدرّب، الجهاز المحلي يحكم.**
2. **الاختبار (test) ما ينلمس أبداً** لاختيار أي شي. كل اختيار على `val` فقط.

**الخطوات:** `Runtime → Change runtime type → GPU (T4)` ثم شغّل الخلايا بالترتيب.

## ١) المكتبات + فحص البطاقة

In [ ]:
!pip -q install medmnist scikit-learn
import torch, torchvision, numpy as np
print("torch      :", torch.__version__)
print("torchvision:", torchvision.__version__)
assert torch.cuda.is_available(), "Runtime -> Change runtime type -> GPU (T4)"
p = torch.cuda.get_device_properties(0)
print("GPU        : %s | %.1f GB" % (p.name, p.total_memory / 1e9))

## ٢) الإعدادات

عدّل هذي الخلية فقط. الافتراضي مضبوط على **`oct_bin`** — المهمة الوحيدة بالمشروع الي
**ماعندها أي رقم لحد الآن** لأنها سقطت مرتين محلياً.

In [ ]:
DATASET   = "octmnist"     # breastmnist | dermamnist | retinamnist | octmnist | pathmnist ...
SIZE      = 224            # 28 | 64 | 128 | 224   (T4 تتحمل 224 مريح)
BINARY    = True           # True = الرأس السريري الثنائي (شوف BINARY_TASKS تحت)
BACKBONE  = "resnet18"     # resnet18 | resnet50 | efficientnet_b0 | convnext_tiny
SUFFIX    = ""             # "_v2" -> يحفظ باسم <key>_v2

EPOCHS    = 30
WARMUP    = 3              # حِقب المرحلة أ (الجذع مجمّد)
BATCH     = 64             # 224px + resnet18 على T4: 64 مريحة
LR        = 3e-4           # المرحلة ب (الشبكة كاملة)
HEAD_LR   = 1e-3           # المرحلة أ (الرأس فقط)
PATIENCE  = 8
DROPOUT   = 0.4
LABEL_SMOOTH = 0.05
MAX_TRAIN = 0              # 0 = كل بيانات التدريب (T4 ما تحتاج تقليص)
VAL_MAX   = 0              # 0 = كل بيانات التحقق
AMP       = True           # على T4 آمنة. (فشلت على GTX 1650 محلياً — مسجّل بالسجل)
SEEDS     = [0]            # [0,1,2,3,4] -> ensemble من ٥ بذور

KEY = DATASET.replace("mnist", "") + ("_bin" if BINARY else "") + SUFFIX
print("KEY =", KEY)

## ٣) المهام الثنائية السريرية

منقولة **حرفياً** من `train_medmnist_v2.py`. هذا مهم: `binary_positive` ينحفظ بالـcheckpoint،
وغيابه هو بالضبط العطب الي خلّى `tune_threshold.py` يتخطّى `breast_v2` (الخطوة ٩ بالسجل).

In [ ]:
BINARY_TASKS = {
    "retinamnist": {
        "name": "referable diabetic retinopathy (grade >= 2)",
        "positive": [2, 3, 4],
        "labels": ["non-referable (grade 0-1)", "referable DR (grade 2-4)"]},
    "dermamnist": {
        "name": "malignant / pre-malignant vs benign skin lesion",
        "positive": [0, 1, 4],
        "labels": ["benign (bkl/df/nv/vasc)", "malignant or pre-malignant (akiec/bcc/mel)"]},
    "octmnist": {
        "name": "retinal disease vs normal OCT",
        "positive": [0, 1, 2],
        "labels": ["normal", "disease (CNV/DME/drusen)"]},
}

# أي فهرس هو "المرض". رؤوس _bin تعيد التوسيم فيصير المرض = 1.
# BreastMNIST استثناء: ترتيبه الأصلي ['malignant','normal, benign'] -> المرض عند 0.
# افتراض الخطأ هنا لا يرمي خطأ — يضبط العتبة على اكتشاف الصحة. (الخطوة ١٤ بالسجل)
CLINICAL_POSITIVE = {"breast_v2": 0, "breast": 0}

if BINARY:
    assert DATASET in BINARY_TASKS, "ماكو مهمة ثنائية معرّفة لـ%s" % DATASET
    print(BINARY_TASKS[DATASET]["name"])

## ٤) البيانات + التحويلات (نفس `train_medmnist_v2.py`)

In [ ]:
import medmnist
from medmnist import INFO
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

IM_MEAN, IM_STD = [0.485, 0.456, 0.406], [0.229, 0.224, 0.225]

def load_split(split):
    DataClass = getattr(medmnist, INFO[DATASET]["python_class"])
    ds = DataClass(split=split, download=True, size=SIZE, root="/content/medmnist")
    return ds.imgs, ds.labels.astype(np.int64).reshape(-1)

def to_binary(y):
    pos = set(BINARY_TASKS[DATASET]["positive"])
    return np.array([1 if int(v) in pos else 0 for v in y], dtype=np.int64)

train_tf = transforms.Compose([
    transforms.ToPILImage(),
    transforms.RandomResizedCrop(SIZE, scale=(0.7, 1.0), ratio=(0.85, 1.18)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(p=0.2),
    transforms.RandomApply([transforms.ColorJitter(0.25, 0.25, 0.15, 0.03)], p=0.7),
    transforms.RandomRotation(20),
    transforms.ToTensor(), transforms.Normalize(IM_MEAN, IM_STD),
    transforms.RandomErasing(p=0.25, scale=(0.02, 0.12)),
])
eval_tf = transforms.Compose([
    transforms.ToPILImage(), transforms.Resize((SIZE, SIZE)),
    transforms.ToTensor(), transforms.Normalize(IM_MEAN, IM_STD),
])

class DS(Dataset):
    def __init__(self, X, y, tf): self.X, self.y, self.tf = X, y, tf
    def __len__(self): return len(self.y)
    def __getitem__(self, i):
        im = self.X[i]
        if im.ndim == 2: im = np.repeat(im[..., None], 3, axis=-1)
        return self.tf(np.ascontiguousarray(im)), int(self.y[i])

Xtr, ytr = load_split("train"); Xva, yva = load_split("val"); Xte, yte = load_split("test")
if BINARY:
    ytr, yva, yte = to_binary(ytr), to_binary(yva), to_binary(yte)
if MAX_TRAIN and len(ytr) > MAX_TRAIN:
    idx = np.random.RandomState(0).choice(len(ytr), MAX_TRAIN, replace=False)
    Xtr, ytr = Xtr[idx], ytr[idx]

classes = (BINARY_TASKS[DATASET]["labels"] if BINARY
           else [INFO[DATASET]["label"][str(i)] for i in range(len(INFO[DATASET]["label"]))])
n_cls = len(classes)
print("train=%d val=%d test=%d | classes=%d" % (len(ytr), len(yva), len(yte), n_cls))
print("test balance:", np.bincount(yte, minlength=n_cls).tolist())

## ٥) الجذع

`resnet18` هو الافتراضي لأنه **يتحمّل مباشرة** بـ`api/nets.py` بدون أي تعديل.
الجذوع الأقوى تحتاج فرع جديد بـ`nets.py` — الخلية الأخيرة تطبعه لك إذا اخترت واحد منها.

In [ ]:
import torch.nn as nn

def build_net(arch, num_classes, dropout, pretrained=True):
    # Returns (net, head_module). The head is what stage A trains alone.
    def head(infeat):
        return (nn.Sequential(nn.Dropout(dropout), nn.Linear(infeat, num_classes))
                if dropout and dropout > 0 else nn.Linear(infeat, num_classes))
    if arch == "resnet18":
        net = torchvision.models.resnet18(weights="IMAGENET1K_V1" if pretrained else None)
        net.fc = head(net.fc.in_features); return net, net.fc
    if arch == "resnet50":
        net = torchvision.models.resnet50(weights="IMAGENET1K_V2" if pretrained else None)
        net.fc = head(net.fc.in_features); return net, net.fc
    if arch == "efficientnet_b0":
        net = torchvision.models.efficientnet_b0(weights="IMAGENET1K_V1" if pretrained else None)
        net.classifier = head(net.classifier[1].in_features); return net, net.classifier
    if arch == "convnext_tiny":
        net = torchvision.models.convnext_tiny(weights="IMAGENET1K_V1" if pretrained else None)
        infeat = net.classifier[2].in_features
        net.classifier[2] = head(infeat); return net, net.classifier[2]
    raise ValueError("unknown backbone: %s" % arch)

_n, _h = build_net(BACKBONE, n_cls, DROPOUT)
print(BACKBONE, "| params = %.1fM" % (sum(p.numel() for p in _n.parameters()) / 1e6))
del _n, _h

## ٦) التدريب — نفس الوصفة بالضبط

مرحلتان (جذع مجمّد ← ضبط كامل)، AdamW + cosine، أوزان أصناف، label smoothing،
**الاختيار على دقة التحقق** (مو الخسارة)، توقف مبكر، وحارس nan.

In [ ]:
import time, copy
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def train_one(seed):
    torch.manual_seed(seed); np.random.seed(seed)
    tl = DataLoader(DS(Xtr, ytr, train_tf), batch_size=BATCH, shuffle=True,
                    num_workers=2, pin_memory=True, drop_last=False)
    vl = DataLoader(DS(Xva, yva, eval_tf), batch_size=128, num_workers=2, pin_memory=True)

    counts = np.bincount(ytr, minlength=n_cls)
    w = counts.sum() / (n_cls * np.maximum(counts, 1))
    net, head = build_net(BACKBONE, n_cls, DROPOUT)
    net = net.cuda()
    crit = nn.CrossEntropyLoss(weight=torch.tensor(w, dtype=torch.float32).cuda(),
                               label_smoothing=LABEL_SMOOTH)
    scaler = torch.amp.GradScaler("cuda", enabled=AMP)
    head_ids = {id(p) for p in head.parameters()}

    def freeze(frozen):
        for p in net.parameters():
            p.requires_grad = (id(p) in head_ids) if frozen else True

    @torch.no_grad()
    def collect(loader, tta=False):
        net.eval(); ys, ps = [], []
        for xb, yb in loader:
            xb = xb.cuda(non_blocking=True)
            with torch.amp.autocast("cuda", enabled=AMP):
                p = torch.softmax(net(xb), 1)
                if tta: p = (p + torch.softmax(net(torch.flip(xb, dims=[3])), 1)) / 2
            ps.append(p.float().cpu().numpy()); ys.append(yb.numpy())
        return np.concatenate(ys), np.concatenate(ps)

    freeze(True)
    opt = torch.optim.AdamW([p for p in net.parameters() if p.requires_grad],
                            lr=HEAD_LR, weight_decay=1e-4)
    stage, sched = "A(head)", None
    best_acc, best_state, bad, history = -1.0, None, 0, []
    t0 = time.time()
    for ep in range(EPOCHS):
        if ep == WARMUP:
            freeze(False)
            opt = torch.optim.AdamW(net.parameters(), lr=LR, weight_decay=1e-4)
            sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, max(1, EPOCHS - WARMUP))
            stage = "B(full)"
        net.train(); tot = 0.0
        for xb, yb in tl:
            xb, yb = xb.cuda(non_blocking=True), yb.cuda(non_blocking=True)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda", enabled=AMP):
                loss = crit(net(xb), yb)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
            tot += loss.item() * len(xb)
        if sched: sched.step()
        yv, pv = collect(vl)
        vacc = accuracy_score(yv, pv.argmax(1))
        print("  ep %2d/%d [%s] loss=%.4f val_acc=%.4f val_bacc=%.4f"
              % (ep + 1, EPOCHS, stage, tot / len(ytr), vacc,
                 balanced_accuracy_score(yv, pv.argmax(1))), flush=True)
        history.append({"epoch": ep + 1, "stage": stage,
                        "train_loss": round(tot / len(ytr), 4),
                        "val_acc": round(float(vacc), 4)})
        if not np.isfinite(tot):
            raise SystemExit("[FATAL] non-finite loss at epoch %d — set AMP=False" % (ep + 1))
        if vacc > best_acc + 1e-4:
            best_acc, best_state, bad = vacc, copy.deepcopy(net.state_dict()), 0
        else:
            bad += 1
            if bad >= PATIENCE:
                print("  [early stop] patience %d" % PATIENCE); break
    net.load_state_dict(best_state)
    return net, history, best_acc, time.time() - t0

nets_trained = []
for s in SEEDS:
    print("\n=== seed %d ===" % s)
    nets_trained.append((s,) + train_one(s))
print("\ntrained %d model(s)" % len(nets_trained))

## ٧) التقييم — TTA يُبقى **فقط إذا ساعد**، والاختبار ينلمس مرة وحدة

In [ ]:
from sklearn.metrics import (roc_auc_score, f1_score, confusion_matrix,
                             classification_report)

vl = DataLoader(DS(Xva, yva, eval_tf), batch_size=128, num_workers=2)
el = DataLoader(DS(Xte, yte, eval_tf), batch_size=128, num_workers=2)

@torch.no_grad()
def probs(net, loader, tta):
    net.eval(); out = []
    for xb, _ in loader:
        xb = xb.cuda(non_blocking=True)
        with torch.amp.autocast("cuda", enabled=AMP):
            p = torch.softmax(net(xb), 1)
            if tta: p = (p + torch.softmax(net(torch.flip(xb, dims=[3])), 1)) / 2
        out.append(p.float().cpu().numpy())
    return np.concatenate(out)

# قرار الـTTA يُتخذ على val فقط
pv_no  = np.mean([probs(n, vl, False) for _, n, _, _, _ in nets_trained], 0)
pv_tta = np.mean([probs(n, vl, True)  for _, n, _, _, _ in nets_trained], 0)
USE_TTA = accuracy_score(yva, pv_tta.argmax(1)) > accuracy_score(yva, pv_no.argmax(1))
print("val no-TTA=%.4f  val TTA=%.4f  -> use_tta=%s"
      % (accuracy_score(yva, pv_no.argmax(1)), accuracy_score(yva, pv_tta.argmax(1)), USE_TTA))

# ---- القياس الوحيد على الاختبار ----
pt = np.mean([probs(n, el, USE_TTA) for _, n, _, _, _ in nets_trained], 0)
pred = pt.argmax(1)
test_acc = accuracy_score(yte, pred)
print("\nTEST accuracy = %.4f" % test_acc)
print(classification_report(yte, pred, target_names=classes, zero_division=0))

try:
    ptn = pt / np.clip(pt.sum(1, keepdims=True), 1e-12, None)   # fp16 softmax لا يجمع 1
    auc = (roc_auc_score(yte, ptn[:, 1]) if n_cls == 2
           else roc_auc_score(yte, ptn, multi_class="ovr", average="macro"))
except Exception as e:
    print("[warn] AUC:", type(e).__name__, e); auc = None
print("TEST AUC =", auc)

## ٨) الحفظ — **بنفس صيغة المشروع بالضبط**

الحقول `medmnist` و`binary_positive` و`tta` تنحفظ صراحةً. غيابها هو الي سبّب عطبين
موثّقين بالسجل (الخطوتان ٩ و١٩).

In [ ]:
import json, shutil, os
os.makedirs("/content/out", exist_ok=True)
best_net = nets_trained[0][1]

ck = {"state_dict": best_net.state_dict(), "size": SIZE, "classes": classes,
      "mean": IM_MEAN, "std": IM_STD, "dropout": DROPOUT,
      "binary_task": BINARY, "tta": bool(USE_TTA),
      "medmnist": DATASET,
      "binary_positive": (BINARY_TASKS[DATASET]["positive"] if BINARY else None),
      "arch": BACKBONE, "trained_on": "colab"}
torch.save(ck, "/content/out/%s.pt" % KEY)

metrics = {
    "model": "%s_%s_colab" % (KEY, BACKBONE), "dataset_key": KEY, "medmnist": DATASET,
    "task": (BINARY_TASKS[DATASET]["name"] if BINARY else "%d-class classification" % n_cls),
    "binary_task": BINARY, "backbone": BACKBONE, "input_size": SIZE,
    "classes": classes, "n_classes": n_cls,
    "n_train": int(len(ytr)), "n_val": int(len(yva)), "n_test": int(len(yte)),
    "test_accuracy": round(float(test_acc), 4),
    "test_balanced_accuracy": round(float(balanced_accuracy_score(yte, pred)), 4),
    "test_macro_f1": round(float(f1_score(yte, pred, average="macro")), 4),
    "test_auc": None if auc is None else round(float(auc), 4),
    "tta_used": bool(USE_TTA),
    "confusion_matrix": confusion_matrix(yte, pred).tolist(),
    "seeds": SEEDS, "n_members": len(nets_trained),
    "epoch_history": nets_trained[0][2],
    "test_split": "full official MedMNIST test split (never subsampled, never tuned on)",
    "trained_on": "google colab",
    "VERIFY_BEFORE_PUBLISHING": ("This number came from Colab. Re-measure the checkpoint "
                                 "locally with verify_retrain_gains.py before putting it in "
                                 "any table — a number from a different evaluation path is "
                                 "not comparable."),
}
json.dump(metrics, open("/content/out/%s_metrics.json" % KEY, "w", encoding="utf-8"),
          ensure_ascii=False, indent=2)

shutil.make_archive("/content/%s_colab" % KEY, "zip", "/content/out")
print("files:", os.listdir("/content/out"))
from google.colab import files
files.download("/content/%s_colab.zip" % KEY)

## ٩) التركيب بالمشروع

1. فك الضغط وحط الملفين بـ`api/models/`:
   - `<KEY>.pt`
   - `<KEY>_metrics.json`

2. **أعِد القياس محلياً — هذي مو خطوة اختيارية:**

   ```bash
   cd api
   VERIFY_DEVICE=cpu venv_gpu/Scripts/python.exe verify_retrain_gains.py
   ```

   إذا الرقم المحلي طابق رقم كولاب → انشره. إذا اختلف → **الرقم المحلي هو الصحيح**،
   ودوّر على سبب الاختلاف قبل أي نشر.

3. اضبط العتبة (للنماذج الثنائية):

   ```bash
   TUNE_DEVICE=cpu venv_gpu/Scripts/python.exe tune_threshold.py <KEY>
   ```

4. سجّل النتيجة بـ`TRAINING_LOG.md`: ليش انعملت، شنو تغيّر، والرقم قبل وبعد.

### إذا اخترت جذع غير `resnet18`

`api/nets.py` يبني `resnet18` فقط، فالـcheckpoint ما راح يتحمّل. شغّل هذي الخلية
وحط الناتج بـ`nets.py`:

In [ ]:
if BACKBONE != "resnet18":
    print('''
# --- أضف هذا لـ api/nets.py ---
def build_colab_backbone(arch: str, num_classes: int, dropout: float = 0.0):
    # Rebuilds a backbone trained in MedMNIST_Colab_Train.ipynb. The checkpoint records
    # its own `arch`, so serving code should read that field and call this.
    import torch.nn as nn, torchvision
    def head(infeat):
        return (nn.Sequential(nn.Dropout(dropout), nn.Linear(infeat, num_classes))
                if dropout and dropout > 0 else nn.Linear(infeat, num_classes))
    if arch == "resnet50":
        net = torchvision.models.resnet50(weights=None); net.fc = head(net.fc.in_features)
    elif arch == "efficientnet_b0":
        net = torchvision.models.efficientnet_b0(weights=None)
        net.classifier = head(net.classifier[1].in_features)
    elif arch == "convnext_tiny":
        net = torchvision.models.convnext_tiny(weights=None)
        net.classifier[2] = head(net.classifier[2].in_features)
    else:
        raise ValueError(arch)
    return net
''')
else:
    print("resnet18 — يتحمّل مباشرة بـ api/nets.py، ماكو شي تسويه")